<a href="https://colab.research.google.com/github/BanJeoker/TestBook/blob/main/ReAct/Langgraph.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install python-dotenv langchain langchain_openai langchain_community pypdf sentence-transformers PyPDF2 pycryptodome tiktoken faiss-gpu langgraph


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.4/50.4 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 63.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.5/294.5 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.3/245.3 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 58.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.5/85.5 MB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.5/106.5 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.9/400.9 kB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 292.2/29

# Read PDF and Build RAG Tool

In [2]:
%%writefile .env
OPENAI_API_KEY=""
from dotenv import load_dotenv
load_dotenv()

Writing .env


In [4]:
import os
from PyPDF2 import PdfReader
from langchain.text_splitter import CharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain import FAISS

In [35]:
def process_text(text):
    text_splitter = CharacterTextSplitter(
        separator="\n",
        chunk_size=2000,
        chunk_overlap=100,
        length_function=len
    )
    chunks = text_splitter.split_text(text)
    embeddings = OpenAIEmbeddings()
    knowledgeBase = FAISS.from_texts(chunks, embeddings)
    return knowledgeBase

pdf_reader = PdfReader('/content/naf23.pdf')
text = ""
for page in pdf_reader.pages:
    text += page.extract_text()

knowledgeBase = process_text(text)

def faiss_search(query):
    return knowledgeBase.similarity_search(query, k=3)

In [8]:

pdf_path = '/content/naf23.pdf'
def extract_text_from_pdf(pdf_path):
    pdf_reader = PdfReader(pdf_path)
    text = ""
    for page in pdf_reader.pages:
        text += page.extract_text()
    return text
auditor_note_text = extract_text_from_pdf(pdf_path)

# single_prompt

In [10]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate

In [11]:
def prompt_run_agent(question, chain, context):
    response = chain.invoke(
        {
            "context": context,
            "question":question
        }
    )
    print(response)

In [12]:
prompt = PromptTemplate.from_template(
    "You are a financial expert and an underwriter working for commerical banking of a bank. Given the following auditor's note that contains audited financial statements of the company: {context}. "
    "Answer this question: {question}."
)

temperature 0.1 vs 0.9, gpt-4-turbo vs gpt-4o

In [13]:
llm_prompt = ChatOpenAI(temperature=0.1, model_name="gpt-4-turbo")
chain_prompt = prompt | llm_prompt
question = "what is National Ataxia Foudnations's total revenue in 2023"
prompt_run_agent(question=question, chain=chain_prompt, context=auditor_note_text)

# output support and revenue combined, not reveune directly

content="The National Ataxia Foundation's total revenue in 2023 was $4,184,787." additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 10757, 'total_tokens': 10778, 'completion_tokens_details': {'audio_tokens': None, 'reasoning_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}}, 'model_name': 'gpt-4-turbo-2024-04-09', 'system_fingerprint': 'fp_68a5bb159e', 'finish_reason': 'stop', 'logprobs': None} id='run-1be60994-ce07-44e7-a809-b2b77557f8c5-0' usage_metadata={'input_tokens': 10757, 'output_tokens': 21, 'total_tokens': 10778}


In [14]:
llm_prompt = ChatOpenAI(temperature=0.9, model_name="gpt-4-turbo")
chain_prompt = prompt | llm_prompt
question = "what is National Ataxia Foudnations's total revenue in 2023"
prompt_run_agent(question=question, chain=chain_prompt, context=auditor_note_text)

# output support and revenue combined, not reveune directly

content="The National Ataxia Foundation's total revenue in 2023 was $4,184,787." additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 10757, 'total_tokens': 10778, 'completion_tokens_details': {'audio_tokens': None, 'reasoning_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}}, 'model_name': 'gpt-4-turbo-2024-04-09', 'system_fingerprint': 'fp_68a5bb159e', 'finish_reason': 'stop', 'logprobs': None} id='run-95cde96f-5586-478a-ac56-41b37a5ebab4-0' usage_metadata={'input_tokens': 10757, 'output_tokens': 21, 'total_tokens': 10778}


In [18]:
llm_prompt = ChatOpenAI(temperature=0.1, model_name="gpt-4o")
chain_prompt = prompt | llm_prompt
question = "what is National Ataxia Foudnations's total revenue in 2023"
prompt_run_agent(question=question, chain=chain_prompt, context=auditor_note_text)

# output support, revenue, net aseeset released from restrictions combined, not reveune directly

content="National Ataxia Foundation's total revenue in 2023 is calculated by summing the total support and total revenue, as well as the net assets released from restrictions. According to the Statements of Activities for the year ended December 31, 2023:\n\n- Total Support: $3,205,136\n- Total Revenue: $979,651\n- Net Assets Released from Restrictions: $975,954\n\nAdding these amounts together gives:\n\n\\[ \\text{Total Revenue} = \\$3,205,136 + \\$979,651 + \\$975,954 = \\$5,160,741 \\]\n\nTherefore, National Ataxia Foundation's total revenue in 2023 is $5,160,741." additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 145, 'prompt_tokens': 10689, 'total_tokens': 10834, 'completion_tokens_details': {'audio_tokens': None, 'reasoning_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-2024-05-13', 'system_fingerprint': 'fp_057232b607', 'finish_reason': 'stop', 'logprobs': None} id='run-4079f30c-f

In [19]:
llm_prompt = ChatOpenAI(temperature=0.9, model_name="gpt-4o")
chain_prompt = prompt | llm_prompt
question = "what is National Ataxia Foudnations's total revenue in 2023"
prompt_run_agent(question=question, chain=chain_prompt, context=auditor_note_text)

# output support and revenue combined, not reveune directly

content='The total revenue for the National Ataxia Foundation (NAF) in 2023 can be found in the "Statements of Activities" section of the financial statements. Here\'s the breakdown:\n\n- Contribution, memorials, and honorariums: $3,083,219\n- In-kind contributions: $121,917\n- Conference income: $264,816\n- Earned income: $570,833\n- Investment income: $144,002\n\nSumming these amounts gives:\n\n\\[ \n\\text{Total Revenue} = 3,083,219 + 121,917 + 264,816 + 570,833 + 144,002 = 4,184,787 \n\\]\n\nThus, National Ataxia Foundation\'s total revenue in 2023 is **$4,184,787**.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 169, 'prompt_tokens': 10689, 'total_tokens': 10858, 'completion_tokens_details': {'audio_tokens': None, 'reasoning_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-2024-05-13', 'system_fingerprint': 'fp_057232b607', 'finish_reason': 'stop', 'logprobs': None} id='run-45cf36

In [24]:
llm_prompt = ChatOpenAI(temperature=0, model_name="gpt-4-turbo")
chain_prompt = prompt | llm_prompt
question = "what is National Ataxia Foudnations's total revenue in 2023"
prompt_run_agent(question=question, chain=chain_prompt, context=auditor_note_text)

# output support and revenue combined, not reveune directly

content="The National Ataxia Foundation's total revenue in 2023 was $4,184,787." additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 10757, 'total_tokens': 10778, 'completion_tokens_details': {'audio_tokens': None, 'reasoning_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}}, 'model_name': 'gpt-4-turbo-2024-04-09', 'system_fingerprint': 'fp_68a5bb159e', 'finish_reason': 'stop', 'logprobs': None} id='run-96f9d2d7-9981-4189-86f2-b4d8e9f452e1-0' usage_metadata={'input_tokens': 10757, 'output_tokens': 21, 'total_tokens': 10778}


currently settle for temerature 0 and gpt-4-turbo

# langgraph

In [22]:
from langchain_core.tools import Tool
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent

In [23]:
tools = [
    Tool(
        name="Search",
        func=faiss_search,
        description="Useful for searching through the document store based on keywords."
    ),
    Tool(name="Lookup",
        func=faiss_search,
        description="Useful for looking up specific information or facts."
    ),
]

In [25]:
def react_run_agent(question, graph):
    inputs = {"messages": [("user", question)]}
    results=graph.stream(inputs, stream_mode="values");

    for s in results:
        message = s["messages"][-1]
        if isinstance(message, tuple):
            print(message)
        else:
            message.pretty_print()

## chunk size 1k

In [ ]:
react_prompt = "You are a financial expert and an underwriter working for commerical banking of a bank."


In [31]:

llm_react = ChatOpenAI(model="gpt-4-turbo", temperature=0)
graph = create_react_agent(llm_react, tools, state_modifier=react_prompt)

question = "what is National Ataxia Foudnations's total revenue in 2023"
react_run_agent(question=question, graph=graph)

================================ Human Message =================================

what is National Ataxia Foudnations's total revenue in 2023
================================== Ai Message ==================================
Tool Calls:
  Lookup (call_jqr2KlLB15NV3PkxLr7fBNay)
 Call ID: call_jqr2KlLB15NV3PkxLr7fBNay
  Args:
    __arg1: National Ataxia Foundation total revenue 2023
================================= Tool Message =================================
Name: Lookup

[Document(metadata={}, page_content="Fundraising 393,619           -                        393,619           \nTotal Supporting Services 770,661           -                        770,661           \nTotal Expenses 4,131,189       -                        4,131,189       \nChange in Net Assets (222,635)          489,031           266,396           \nBeginning Net Assets 2,607,446       1,552,775       4,160,221       \nEnding Net Assets 2,384,811$     2,041,806$      4,426,617$      2022National Ataxia Foundation\nSt

retrieved 3 documents, one part of the income is missing due to chunking. The model is calcuting the combinded values of support and revenue, not just revenue

In [32]:

llm_react = ChatOpenAI(model="gpt-4o", temperature=0)
graph = create_react_agent(llm_react, tools, state_modifier=react_prompt)

question = "what is National Ataxia Foudnations's total revenue in 2023"
react_run_agent(question=question, graph=graph)

================================ Human Message =================================

what is National Ataxia Foudnations's total revenue in 2023
================================== Ai Message ==================================
Tool Calls:
  Search (call_nIJCZXzeeINhQSPbRJRN7t65)
 Call ID: call_nIJCZXzeeINhQSPbRJRN7t65
  Args:
    __arg1: National Ataxia Foundation total revenue 2023
================================= Tool Message =================================
Name: Search

[Document(metadata={}, page_content="Fundraising 393,619           -                        393,619           \nTotal Supporting Services 770,661           -                        770,661           \nTotal Expenses 4,131,189       -                        4,131,189       \nChange in Net Assets (222,635)          489,031           266,396           \nBeginning Net Assets 2,607,446       1,552,775       4,160,221       \nEnding Net Assets 2,384,811$     2,041,806$      4,426,617$      2022National Ataxia Foundation\nSt

In [33]:

llm_react = ChatOpenAI(model="gpt-4o", temperature=0)
graph = create_react_agent(llm_react, tools, state_modifier=react_prompt)
question = "what is total revenue in 2023"
react_run_agent(question=question, graph=graph)

================================ Human Message =================================

what is total revenue in 2023
================================== Ai Message ==================================
Tool Calls:
  Search (call_8OxZHcTuTEx1MnmlIQxXXrGr)
 Call ID: call_8OxZHcTuTEx1MnmlIQxXXrGr
  Args:
    __arg1: total revenue in 2023
================================= Tool Message =================================
Name: Search

[Document(metadata={}, page_content='Total Revenue 979,651           -                        979,651           \nNet Assets Released from Restrictions 975,954           (975,954)          -                        \nTotal Support and Revenue 3,546,026       638,761           4,184,787       \nExpenses\nProgram services\nResearch 1,574,354       -                        1,574,354       \nEducation and service 1,146,980       -                        1,146,980       \nDrug Development Collaborative 1,188,427       -                        1,188,427       \nTotal Program Se

In [34]:

llm_react = ChatOpenAI(model="gpt-4-turbo", temperature=0)
graph = create_react_agent(llm_react, tools, state_modifier=react_prompt)

question = "what is total revenue in 2023"
react_run_agent(question=question, graph=graph)

================================ Human Message =================================

what is total revenue in 2023
================================== Ai Message ==================================
Tool Calls:
  Lookup (call_z8QkGhdRTpkHlrLpKm0c39rH)
 Call ID: call_z8QkGhdRTpkHlrLpKm0c39rH
  Args:
    __arg1: total revenue in 2023
================================= Tool Message =================================
Name: Lookup

[Document(metadata={}, page_content='Total Revenue 979,651           -                        979,651           \nNet Assets Released from Restrictions 975,954           (975,954)          -                        \nTotal Support and Revenue 3,546,026       638,761           4,184,787       \nExpenses\nProgram services\nResearch 1,574,354       -                        1,574,354       \nEducation and service 1,146,980       -                        1,146,980       \nDrug Development Collaborative 1,188,427       -                        1,188,427       \nTotal Program Se

depends on the query and the way RAG works
* "what is National Ataxia Foudnations's total revenue in 2023" will return documents related to **NAF**

* "what is total revenue in 2023" will return documents that are correct.

the aboved are examples for chunking size 1000.

## chunk size 2k

In [36]:

llm_react = ChatOpenAI(model="gpt-4o", temperature=0)
graph = create_react_agent(llm_react, tools, state_modifier=react_prompt)

question = "what is National Ataxia Foudnations's total revenue in 2023"
react_run_agent(question=question, graph=graph)

================================ Human Message =================================

what is National Ataxia Foudnations's total revenue in 2023
================================== Ai Message ==================================
Tool Calls:
  Search (call_UqMsPbwBJKckumqS8BkiWfW0)
 Call ID: call_UqMsPbwBJKckumqS8BkiWfW0
  Args:
    __arg1: National Ataxia Foundation total revenue 2023
================================= Tool Message =================================
Name: Search

[Document(metadata={}, page_content="Net Assets Released from Restrictions 1,278,770       (1,278,770)      -                        \n \nTotal Support and Revenue 3,908,554       489,031           4,397,585       \nExpenses\nProgram services\nResearch 1,837,520       -                        1,837,520       \nEducation and service 579,104           -                        579,104           \nDrug Development Collaborative 943,904           -                        943,904           \nTotal Program Services 3,360,528

the answer is in the returned document, just still return both support and revenue combined

In [37]:

llm_react = ChatOpenAI(model="gpt-4o", temperature=0)
graph = create_react_agent(llm_react, tools, state_modifier=react_prompt)
question = "what is total revenue in 2023"
react_run_agent(question=question, graph=graph)

================================ Human Message =================================

what is total revenue in 2023
================================== Ai Message ==================================
Tool Calls:
  Search (call_W6M4mLKHKtbZeOcrhf1QMRBN)
 Call ID: call_W6M4mLKHKtbZeOcrhf1QMRBN
  Args:
    __arg1: total revenue in 2023
================================= Tool Message =================================
Name: Search

[Document(metadata={}, page_content="Total Lease Liabilities 58,103 $          \n \n19 \n National Ataxia Foundation  \nNotes to the Financial Statements  \nDecember 31, 2023  and 2022  \n \nNote 6: Net Assets with Donor Restrictions  \n \nNet assets with donor restrictions were available for the following purposes at December 31:  \n \n2023 2022\nSCA2 Fund 880,250 $        500,000 $        \nMR Imaging Study Fund 612,044           -                        \nGeneral Research (Includes stock) 394,144           385,387           \nCSO Funding 360,800           450,588     

In [39]:
llm_prompt = ChatOpenAI(temperature=0, model_name="gpt-4o")
chain_prompt = prompt | llm_prompt
question = "what is total revenue in 2023"
prompt_run_agent(question=question, chain=chain_prompt, context=auditor_note_text)


content='The total revenue for the National Ataxia Foundation in 2023 is $979,651. This is derived from the following components:\n\n- Conference income: $264,816\n- Earned income: $570,833\n- Investment income: $144,002\n\nAdding these amounts together gives the total revenue:\n\n\\[ 264,816 + 570,833 + 144,002 = 979,651 \\]\n\nTherefore, the total revenue for 2023 is $979,651.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 102, 'prompt_tokens': 10680, 'total_tokens': 10782, 'completion_tokens_details': {'audio_tokens': None, 'reasoning_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-2024-05-13', 'system_fingerprint': 'fp_057232b607', 'finish_reason': 'stop', 'logprobs': None} id='run-efd1ef80-799c-44a4-af15-ff7ab4625ff5-0' usage_metadata={'input_tokens': 10680, 'output_tokens': 102, 'total_tokens': 10782}


side by side comparison shows that using a single prompt without ReAct also works

## settle on gpt-4o, chunk size 2k, more advanced questions

In [41]:
question

'what is total revenue in 2023'

In [45]:
question='how much is an item can it be considered to be a equipment'
llm_react = ChatOpenAI(model="gpt-4o", temperature=0)
graph = create_react_agent(llm_react, tools, state_modifier=react_prompt)
react_run_agent(question=question, graph=graph) # gpt-4o

================================ Human Message =================================

how much is an item can it be considered to be a equipment
================================== Ai Message ==================================

The classification of an item as equipment typically depends on its use, value, and the context in which it is being considered. Here are some general guidelines:

1. **Use**: Equipment is usually defined as tangible property that is used in the operations of a business. This includes items that are used to produce goods or services, such as machinery, tools, and vehicles.

2. **Value**: Equipment often has a significant value and is expected to be used over a long period of time. The specific value threshold can vary depending on the business or industry, but it is generally higher than the threshold for supplies or inventory.

3. **Lifespan**: Equipment is typically expected to have a useful life of more than one year. Items that are consumed or used up within a ye

In [46]:
question='how much is an item can it be considered to be a equipment in National Ataxia Foudnations accounting rules'
llm_react = ChatOpenAI(model="gpt-4o", temperature=0)
graph = create_react_agent(llm_react, tools, state_modifier=react_prompt)
react_run_agent(question=question, graph=graph) # gpt-4o

================================ Human Message =================================

how much is an item can it be considered to be a equipment in National Ataxia Foudnations accounting rules
================================== Ai Message ==================================
Tool Calls:
  Search (call_Z7JRiTIIiw7BhyfhKBIVUs3M)
 Call ID: call_Z7JRiTIIiw7BhyfhKBIVUs3M
  Args:
    __arg1: National Ataxia Foundation accounting rules equipment item cost
================================= Tool Message =================================
Name: Search

[Document(metadata={}, page_content='for request from non -participating investigators for approved research projects.  \n  \n13 \n National Ataxia Foundation  \nNotes to the Financial Statements  \nDecember 31, 2023  and 2022  \n \nNote 1:  Summary of Significant Accounting Policies (Continued)  \n \nB. Basis of Accounting and Presentation  \n \nThe accompanying financial statements have been prepared using  the accrual basis of accounting in accordance

In [47]:
question='analyse the changes in total expense, check for any irregular or significant pattern'
llm_react = ChatOpenAI(model="gpt-4o", temperature=0)
graph = create_react_agent(llm_react, tools, state_modifier=react_prompt)
react_run_agent(question=question, graph=graph) # gpt-4o

================================ Human Message =================================

analyse the changes in total expense, check for any irregular or significant pattern
================================== Ai Message ==================================

To analyze the changes in total expense and check for any irregular or significant patterns, I will need to review the relevant financial documents or data. I will search for documents or records that contain information about total expenses over a period of time.

Let's start by searching for documents related to total expenses.
Tool Calls:
  Search (call_FLABAQykghoUlAhr3z7gQQZH)
 Call ID: call_FLABAQykghoUlAhr3z7gQQZH
  Args:
    __arg1: total expenses
================================= Tool Message =================================
Name: Search

[Document(metadata={}, page_content='Total Salaries and Related Expenses 577,767           442,121             188,115           1,208,003       169,242           354,161           1,731,406      

4,996,458 is 2023, 4,131,189 is 2022, again, it is about how things are parsed. 2023 is on one page, 2022 is on another page. The model does not have the ability to connect them toghther

#### adding the few shot examples

In [49]:
react_prompt = f"""
Question: What is the elevation range for the area that the eastern sector of the Colorado orogeny extends into?
Thought: I need to search Colorado orogeny, find the area that the eastern sector of the Colorado orogeny extends into, then find the elevation range of the area.
Action: Search[Colorado orogeny]
Observation: The Colorado orogeny was an episode of mountain building (an orogeny) in Colorado and surrounding areas.
Thought: It does not mention the eastern sector. So I need to look up eastern sector.
Action: Lookup[eastern sector]
Observation: (Result 1 / 1) The eastern sector extends into the High Plains and is called the Central Plains orogeny.
Thought: The eastern sector of Colorado orogeny extends into the High Plains. So I need to search High Plains and find its elevation range.
Action: Search[High Plains]
Observation: High Plains refers to one of two distinct land regions
Thought: I need to instead search High Plains (United States).
Action: Search[High Plains (United States)]
Observation: The High Plains are a subregion of the Great Plains. From east to west, the High Plains rise in elevation from around 1,800 to 7,000 ft (550 to 2,130 m).[3]
Thought: High Plains rise in elevation from around 1,800 to 7,000 ft, so the answer is 1,800 to 7,000 ft.
Action: Finish[1,800 to 7,000 ft]

Question: Musician and satirist Allie Goertz wrote a song about the "The Simpsons" character Milhouse, who Matt Groening named after who?
Thought: The question simplifies to "The Simpsons" character Milhouse is named after who. I only need to search Milhouse and find who it is named after.
Action: Search[Milhouse]
Observation: Milhouse Mussolini Van Houten is a recurring character in the Fox animated television series The Simpsons voiced by Pamela Hayden and created by Matt Groening.
Thought: The paragraph does not tell who Milhouse is named after, maybe I can look up "named after".
Action: Lookup[named after]
Observation: (Result 1 / 1) Milhouse was named after U.S. president Richard Nixon, whose middle name was Milhous.
Thought: Milhouse was named after U.S. president Richard Nixon, so the answer is Richard Nixon.
Action: Finish[Richard Nixon]

Question: Which documentary is about Finnish rock groups, Adam Clayton Powell or The Saimaa Gesture?
Thought: I need to search Adam Clayton Powell and The Saimaa Gesture, and find which documentary is about Finnish rock groups.
Action: Search[Adam Clayton Powell]
Observation: Could not find [Adam Clayton Powell]. Similar: [’Adam Clayton Powell III’, ’Seventh Avenue (Manhattan)’, ’Adam Clayton Powell Jr. State Office Building’, ’Isabel Washington Powell’, ’Adam Powell’, ’Adam Clayton Powell (film)’, ’Giancarlo Esposito’].
Thought: To find the documentary, I can search Adam Clayton Powell (film).
Action: Search[Adam Clayton Powell (film)]
Observation: Adam Clayton Powell is a 1989 American documentary film directed by Richard Kilberg. The film is about the rise and fall of influential African-American politician Adam Clayton Powell Jr.[3][4] It was later aired as part of the PBS series The American Experience.
Thought: Adam Clayton Powell (film) is a documentary about an African-American politician, not Finnish rock groups. So the documentary about Finnish rock groups must instead be The Saimaa Gesture.
Action: Finish[The Saimaa Gesture]

Question: What profession does Nicholas Ray and Elia Kazan have in common?
Thought: I need to search Nicholas Ray and Elia Kazan, find their professions, then find the profession they have in common.
Action: Search[Nicholas Ray]
Observation: Nicholas Ray (born Raymond Nicholas Kienzle Jr., August 7, 1911 - June 16, 1979) was an American film director, screenwriter, and actor best known for the 1955 film Rebel Without a Cause.
Thought: Professions of Nicholas Ray are director, screenwriter, and actor. I need to search Elia Kazan next and find his professions.
Action: Search[Elia Kazan]
Observation: Elia Kazan was an American film and theatre director, producer, screenwriter and actor.
Thought: Professions of Elia Kazan are director, producer, screenwriter, and actor. So profession Nicholas Ray and Elia Kazan have in common is director, screenwriter, and actor.
Action: Finish[director, screenwriter, actor]

Question: Which magazine was started first Arthur’s Magazine or First for Women?
Thought: I need to search Arthur’s Magazine and First for Women, and find which was started first.
Action: Search[Arthur’s Magazine]
Observation: Arthur’s Magazine (1844-1846) was an American literary periodical published in Philadelphia in the 19th century.
Thought: Arthur’s Magazine was started in 1844. I need to search First for Women next.
Action: Search[First for Women]
Observation: First for Women is a woman’s magazine published by Bauer Media Group in the USA.[1] The magazine was started in 1989.
Thought: First for Women was started in 1989. 1844 (Arthur’s Magazine) < 1989 (First for Women), so Arthur’s Magazine was started first.
Action: Finish[Arthur’s Magazine]

Now, you are a financial expert and an underwriter working for commerical banking of a bank.

"""

question='analyse the changes in total expense for National Ataxia Foundation, check for any irregular or significant pattern'
llm_react = ChatOpenAI(model="gpt-4o", temperature=0)
graph = create_react_agent(llm_react, tools, state_modifier=react_prompt)
react_run_agent(question=question, graph=graph) # gpt-4o

================================ Human Message =================================

analyse the changes in total expense for National Ataxia Foundation, check for any irregular or significant pattern
================================== Ai Message ==================================
Tool Calls:
  Search (call_Xlb1XH1pwfF39S42aP6Ikb8C)
 Call ID: call_Xlb1XH1pwfF39S42aP6Ikb8C
  Args:
    __arg1: National Ataxia Foundation total expense
  Lookup (call_MpXYO2nnuyGPqKtvUk2d5oIO)
 Call ID: call_MpXYO2nnuyGPqKtvUk2d5oIO
  Args:
    __arg1: National Ataxia Foundation
================================= Tool Message =================================
Name: Lookup

[Document(metadata={}, page_content='e-newsletter devoted to ataxia related articles including the latest research and other information beneficial to \nindividuals with ataxia. The Foundation also offers several webinars throughout the year on current topics relevant to \nthe ataxia community. Other educational materials, such as books, vide

4,996,458 is 2023, 4,131,189 is 2022; detailed numbers are not correct

In [51]:
question="why National Ataxia Foundation's total assets decrease so much from 2023 to 2022"
llm_react = ChatOpenAI(model="gpt-4o", temperature=0)
graph = create_react_agent(llm_react, tools, state_modifier=react_prompt)
react_run_agent(question=question, graph=graph) # gpt-4o

================================ Human Message =================================

why National Ataxia Foundation's total assets decrease so much from 2023 to 2022
================================== Ai Message ==================================
Tool Calls:
  Search (call_Auks64QxxATZbupK1RO5isYr)
 Call ID: call_Auks64QxxATZbupK1RO5isYr
  Args:
    __arg1: National Ataxia Foundation total assets decrease 2023 to 2022
================================= Tool Message =================================
Name: Search

[Document(metadata={}, page_content="Net Assets Released from Restrictions 1,278,770       (1,278,770)      -                        \n \nTotal Support and Revenue 3,908,554       489,031           4,397,585       \nExpenses\nProgram services\nResearch 1,837,520       -                        1,837,520       \nEducation and service 579,104           -                        579,104           \nDrug Development Collaborative 943,904           -                        943,904        

In [52]:
question="why total assets decreases so much from 2023 to 2022"
llm_react = ChatOpenAI(model="gpt-4o", temperature=0)
graph = create_react_agent(llm_react, tools, state_modifier=react_prompt)
react_run_agent(question=question, graph=graph) # gpt-4o

================================ Human Message =================================

why total assets decreases so much from 2023 to 2022
================================== Ai Message ==================================

To determine why total assets decreased significantly from 2023 to 2022, I would need to review the financial statements and any accompanying notes or management discussion and analysis (MD&A) for those years. These documents typically provide detailed explanations for significant changes in financial metrics.

Please provide the financial statements or specify the company in question so I can look up the relevant information.


In [53]:
question="why awards does National Ataxia Foundation give each year"
llm_react = ChatOpenAI(model="gpt-4o", temperature=0)
graph = create_react_agent(llm_react, tools, state_modifier=react_prompt)
react_run_agent(question=question, graph=graph) # gpt-4o

================================ Human Message =================================

why awards does National Ataxia Foundation give each year
================================== Ai Message ==================================
Tool Calls:
  Search (call_MMKUiWPFxG8TnY7tzp8LG0sx)
 Call ID: call_MMKUiWPFxG8TnY7tzp8LG0sx
  Args:
    __arg1: National Ataxia Foundation awards
================================= Tool Message =================================
Name: Search

[Document(metadata={}, page_content='ataxia. Annual grants are selected through a competitive review process that includes NAF’s Medical and Research \nAdvisory Board members and over 80 leading ataxia scien tists and clinicians, both domestic and international.  \n \nEach year NAF awards grants to researchers from domestic and international non -profit and for -profit institutions that \ndirectly support basic and translational ataxia research through the following funding mechanisms:  \n \n1. Pioneer SCA3 Translational Research A

In [56]:
question="how can a scientific Ph.D researcher be associtaed with National Ataxia Foundation financially, what awarads or funding can he/she receives"
llm_react = ChatOpenAI(model="gpt-4o", temperature=0)
graph = create_react_agent(llm_react, tools, state_modifier=react_prompt)
react_run_agent(question=question, graph=graph) # gpt-4o

================================ Human Message =================================

how can a scientific Ph.D researcher be associtaed with National Ataxia Foundation financially, what awarads or funding can he/she receives
================================== Ai Message ==================================
Tool Calls:
  Search (call_CUmmfKQDfVmi6IxRGadlDcjB)
 Call ID: call_CUmmfKQDfVmi6IxRGadlDcjB
  Args:
    __arg1: National Ataxia Foundation awards funding Ph.D researcher
================================= Tool Message =================================
Name: Search

[Document(metadata={}, page_content='position and increase chances to establish an independent ataxia research progra m. \n \n5. Pre-doctoral Fellowship to Promote Diversity in Ataxia Research: Merit -based award intended to enhance \nresearch and/or clinical training of promising graduate students from historically underrepresented \nbackgrounds who are matriculated in pre -doctoral or c linical health professional degree trai

In [57]:
question="how can a scientific Ph.D researcher be associtaed with National Ataxia Foundation financially"
llm_react = ChatOpenAI(model="gpt-4o", temperature=0)
graph = create_react_agent(llm_react, tools, state_modifier=react_prompt)
react_run_agent(question=question, graph=graph) # gpt-4o

================================ Human Message =================================

how can a scientific Ph.D researcher be associtaed with National Ataxia Foundation financially
================================== Ai Message ==================================

A scientific Ph.D. researcher can be financially associated with the National Ataxia Foundation (NAF) through various funding opportunities and grants that the foundation offers. Here are some common ways:

1. **Research Grants**: The NAF provides research grants to support studies related to ataxia. Researchers can apply for these grants to fund their projects.

2. **Fellowships**: The NAF offers fellowships for postdoctoral researchers and early-career scientists. These fellowships provide financial support for researchers working on ataxia-related projects.

3. **Collaborative Research Funding**: The NAF may fund collaborative research projects that involve multiple institutions or researchers working together on ataxia research

you can see the llm escape from using our defined tools

## use old agent

In [58]:
from langchain import hub
from langchain_openai import OpenAI
from langchain.agents import create_react_agent
from langchain.agents import AgentExecutor

In [64]:
prompt = hub.pull("hwchase17/react")

# already defined
# tools = [
#     Tool(
#         name="Search",
#         func=faiss_search,
#         description="Useful for searching through the document store based on keywords."
#     ),
#     Tool(name="Lookup",
#         func=faiss_search,
#         description="Useful for looking up specific information or facts."
#     ),
# ]

questio_heading="You are a financial expert and an underwriter working for commerical banking of a bank. "
def old_react_run(question, agent_executor):
    result = agent_executor.invoke({"input": questio_heading+ question})
    print(result)




/usr/local/lib/python3.10/dist-packages/langsmith/client.py:323: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


In [65]:
llm = OpenAI(temperature=0)
agent = create_react_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True, handle_parsing_errors=True)

questions="what is National Ataxia Foudnations's revenue 2023"
old_react_run(question=question,agent_executor=agent_executor)



> Entering new AgentExecutor chain...
 I should search for information on how scientific Ph.D researchers can be associated with the National Ataxia Foundation financially.
Action: Search
Action Input: "scientific Ph.D researchers National Ataxia Foundation financial association"[Document(metadata={}, page_content='ataxia. Annual grants are selected through a competitive review process that includes NAF’s Medical and Research \nAdvisory Board members and over 80 leading ataxia scien tists and clinicians, both domestic and international.  \n \nEach year NAF awards grants to researchers from domestic and international non -profit and for -profit institutions that \ndirectly support basic and translational ataxia research through the following funding mechanisms:  \n \n1. Pioneer SCA3 Translational Research Awards: Annually granted to outstanding research proposals that aim \nto make significant advancements in the development of treatments and/or improvements to patient care for \nSpin

BadRequestError: Error code: 400 - {'error': {'message': "This model's maximum context length is 4097 tokens, however you requested 4523 tokens (4267 in your prompt; 256 for the completion). Please reduce your prompt; or completion length.", 'type': 'invalid_request_error', 'param': None, 'code': None}}